# Phase 2 - Part 2: Wind farm optimisation (simple, on synthetic wind)

Optimise an offshore North Sea farm with the Phase 2 stack. Capacity and turbine are fixed, so maximising AEP = maximising **capacity factor**; LCOE is a secondary diagnostic. 

Scorer constraints: **exactly 55 × IEA_22MW** (1210 MW); centre over sea in the allowed zone, **water depth ≤ 50 m** (fixed-bottom); all turbines in the 15×15 km box (`|x|,|y| ≤ 7500 m`); spacing ≥ 5 rotor diameters (≥ 1420 m). Ranked on **capacity factor**.

In [ ]:
import os, sys, warnings
from pathlib import Path
try:
    _here = Path(__vsc_ipynb_file__).resolve().parent   # VS Code sets this
except NameError:
    _here = Path.cwd().resolve()
_root = next(d for d in [_here, *_here.parents]
             if (d / 'part0_dataset_setup' / 'target_loader.py').exists())
os.chdir(_root)
sys.path[:0] = ['.', 'part0_dataset_setup', 'part1_forecast',
                'part2_siting', 'part3_economics']
warnings.filterwarnings('ignore')

import numpy as np, pandas as pd
import synthetic_generator as sg
import zone, synth_wind
import optimization as opt
import cost_model
from wind_farm_simulator import grid_layout
from turbines_catalog import get_spec, CATALOG

print("Imports OK. Catalog:", list(CATALOG))

## 1. Synthetic year, wind cache, and zone bounds

In [ ]:
hist = sg.load_coarse_history()
model = sg.fit_ar_generator(hist)
synth = sg.sample_ar_year(model, seed=0)
cache = synth_wind.SynthWindCache(synth, hub_height_m=170.0)

(lat_b, lon_b) = zone.zone_bounds()
print("zone bounds:", lat_b, lon_b)


## 2. Joint optimisation (placement × layout; turbine fixed to IEA 22 MW)

In [ ]:
BOX_M = 15000.0   # full side; half-extent 7500 m == scorer's |x|,|y| <= 7500

res = opt.optimize_joint(
    initial_centre_lat=float(np.mean(lat_b)), initial_centre_lon=float(np.mean(lon_b)),
    initial_turbine_key="IEA_22MW", n_turbines=55,
    wind_cache=cache, lat_bounds=lat_b, lon_bounds=lon_b,
    box_size_m=BOX_M, max_turbines=55, min_spacing_d=5.0,
    is_allowed=lambda la, lo: zone.is_in_allowed_zone(la, lo, max_depth_m=50),
    fix_turbine=True, n_outer_iters=2, max_iter_per_axis=10, seed=0)

best = res.best_config
print(f"AEP={res.best_aep_gwh:.1f} GWh | CF={res.best_capacity_factor:.3f} | "
      f"wake={res.best_wake_loss:.3f} | turbine={best.turbine_key} | "
      f"centre=({best.centre_lat:.3f},{best.centre_lon:.3f}) | n={best.n_turbines()}")
print(f"evaluations={res.n_evaluations} | elapsed={res.elapsed_seconds:.0f}s")
assert zone.is_in_allowed_zone(best.centre_lat, best.centre_lon), "centre left the allowed zone"

## 3. LCOE diagnostic

In [ ]:
spec = get_spec(best.turbine_key)
capacity_mw = best.n_turbines() * spec.rated_power_mw

cost = cost_model.evaluate_farm(
    capacity_mw=capacity_mw,
    n_turbines=best.n_turbines(),
    aep_gwh=res.best_aep_gwh,
)
print(cost.summary())
print()
print(f"CAPEX:  {cost.capex_eur/1e6:,.0f} M EUR")
print(f"OPEX:   {cost.opex_eur_per_year/1e6:,.1f} M EUR/year")
print(f"LCOE:   {cost.lcoe_eur_per_mwh:.1f} EUR/MWh "
      f"(capex {cost.lcoe_components['capex']:.1f} + opex {cost.lcoe_components['opex']:.1f})")


## 3b. Visualiser le layout optimise

In [ ]:
import matplotlib.pyplot as plt
spec_b = get_spec(best.turbine_key)
fig, ax = plt.subplots(figsize=(5, 5))
half = BOX_M / 2
ax.add_patch(plt.Rectangle((-half, -half), BOX_M, BOX_M, fill=False,
                           ec="tab:red", ls="--", label="boîte 15×15 km"))
ax.scatter(best.layout_x_m, best.layout_y_m, s=40, c="tab:blue",
           label=f"{best.n_turbines()}× {best.turbine_key}")
ax.set_aspect("equal"); ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
ax.set_title(f"Layout optimisé @ ({best.centre_lat:.2f}, {best.centre_lon:.2f})  "
             f"AEP={res.best_aep_gwh:.0f} GWh")
ax.legend(loc="upper right", fontsize=8); plt.tight_layout(); plt.show()


## 4. Export the submission

In [ ]:
opt.export_submission(best, "part2_siting/submission.json", team="baseline")
print("wrote submission.json")

import json
print(json.dumps({k: (v if not isinstance(v, list) else f"<{len(v)} pts>")
                  for k, v in json.loads(Path("part2_siting/submission.json").read_text()).items()}, indent=2))
